# IEEE-33 Graph Transformer DNR Surrogate Training

This notebook trains surrogate models for Distribution Network Reconfiguration (DNR) using:

```text
data/ieee33_dnr_ranking_dataset.h5
```

## Research objective

We are **not** solving topology classification. We solve candidate scoring/ranking:

```text
(load scenario + candidate topology) -> total_loss_kw, min_voltage_pu, max_line_loading_percent
```

At inference, the surrogate evaluates all candidate topologies for a load scenario, filters infeasible candidates, and selects the minimum predicted loss topology.

## Models implemented

1. Global Feature MLP
2. GCN
3. GAT
4. Graph Transformer

## Critical split rule

Splits are performed by **scenario only**:

- Train: 70%
- Validation: 15%
- Test: 15%

Candidate graphs from the same scenario never appear in multiple splits.


In [ ]:
import os
import time
import math
import json
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import h5py
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import pandapower as pp
import pandapower.networks as pn
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

try:
    from torch_geometric.data import Data, Batch
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GCNConv, GATv2Conv, TransformerConv, global_mean_pool, global_max_pool
    PYG_AVAILABLE = True
except Exception as exc:
    PYG_AVAILABLE = False
    PYG_IMPORT_ERROR = exc

SEED = 20260623
DATA_PATH = "data/ieee33_dnr_ranking_dataset.h5"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

BATCH_SIZE = 512
NUM_WORKERS = 0
MAX_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# Set to an integer for quick debugging; keep None for the full dataset.
DEBUG_MAX_SCENARIOS = None
MODELS_TO_TRAIN = ["MLP", "GCN", "GAT", "GraphTransformer"]

TARGET_COLUMNS = ["total_loss_kw", "min_voltage_pu", "max_line_loading_percent"]
FULL_LABEL_COLUMNS = [
    "total_loss_kw", "min_voltage_pu", "max_line_loading_percent",
    "voltage_violation_count", "overload_count", "constraint_violation_count",
]
TARGET_WEIGHTS = torch.tensor([1.0, 2.0, 1.0], dtype=torch.float32)

FEASIBLE_VOLTAGE_MIN = 0.90
FEASIBLE_LOADING_MAX = 100.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not PYG_AVAILABLE:
    raise ImportError(f"torch_geometric is required for this notebook. Original import error: {PYG_IMPORT_ERROR}")

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"PyG available: {PYG_AVAILABLE}")
print(f"Dataset path: {DATA_PATH}")


## 1. Dataset inspection and scenario-only split

The loader keeps scenario boundaries intact. A scenario contributes all candidate topologies to exactly one split.


In [ ]:
def list_scenarios(h5_path: str) -> List[str]:
    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"Dataset not found: {h5_path}. Run the generation notebook first.")
    with h5py.File(h5_path, "r") as h5:
        scenarios = sorted(name for name in h5.keys() if name.startswith("scenario_"))
    if DEBUG_MAX_SCENARIOS is not None:
        scenarios = scenarios[:int(DEBUG_MAX_SCENARIOS)]
    if len(scenarios) == 0:
        raise RuntimeError("No scenario groups found in HDF5 dataset.")
    return scenarios


def inspect_dataset(h5_path: str):
    scenarios = list_scenarios(h5_path)
    with h5py.File(h5_path, "r") as h5:
        first = h5[scenarios[0]]
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        print("Dataset inspection")
        print("-" * 28)
        print(f"Scenarios              : {len(scenarios):,}")
        print(f"candidate_node_features: {first['candidate_node_features'].shape}")
        print(f"candidate_edge_features: {first['candidate_edge_features'].shape}")
        print(f"candidate_global_features: {first['candidate_global_features'].shape}")
        print(f"candidate_labels       : {first['candidate_labels'].shape}")
        print(f"candidate_ranks        : {first['candidate_ranks'].shape}")
        print(f"topology_pool_size     : {attrs.get('topology_pool_size', 'unknown')}")
    return scenarios


def split_scenarios(scenarios: List[str], seed: int = SEED):
    rng = np.random.default_rng(seed)
    shuffled = np.array(scenarios, dtype=object)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(round(TRAIN_RATIO * n))
    n_val = int(round(VAL_RATIO * n))
    train = list(shuffled[:n_train])
    val = list(shuffled[n_train:n_train + n_val])
    test = list(shuffled[n_train + n_val:])
    return train, val, test


def verify_no_leakage(train: List[str], val: List[str], test: List[str]):
    train_set, val_set, test_set = set(train), set(val), set(test)
    overlaps = {
        "train_val": len(train_set & val_set),
        "train_test": len(train_set & test_set),
        "val_test": len(val_set & test_set),
    }
    assert all(value == 0 for value in overlaps.values()), f"Scenario leakage detected: {overlaps}"
    print("Split verification")
    print("-" * 24)
    print(f"Train scenarios: {len(train):,}")
    print(f"Val scenarios  : {len(val):,}")
    print(f"Test scenarios : {len(test):,}")
    print(f"Overlaps       : {overlaps}")

SCENARIOS = inspect_dataset(DATA_PATH)
TRAIN_SCENARIOS, VAL_SCENARIOS, TEST_SCENARIOS = split_scenarios(SCENARIOS, SEED)
verify_no_leakage(TRAIN_SCENARIOS, VAL_SCENARIOS, TEST_SCENARIOS)


## 2. IEEE-33 physical edge index and HDF5-backed PyG dataset

Candidate edge features include all 37 physical lines with a switch-status column. For message passing, the dataset constructs active undirected edges per candidate from `switch_status > 0.5`.


In [ ]:
STANDARD_TIE_BRANCH_DATA = [
    {"from_bus": 20, "to_bus": 7, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 8, "to_bus": 14, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 11, "to_bus": 21, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 17, "to_bus": 32, "r_ohm": 0.5, "x_ohm": 0.5},
    {"from_bus": 24, "to_bus": 28, "r_ohm": 0.5, "x_ohm": 0.5},
]


def ensure_line_columns(net):
    if "in_service" not in net.line.columns:
        net.line["in_service"] = True
    net.line["in_service"] = net.line["in_service"].fillna(True).astype(bool)
    if "is_tie" not in net.line.columns:
        net.line["is_tie"] = False
    net.line["is_tie"] = net.line["is_tie"].fillna(False).astype(bool)


def line_mask_for_endpoints(net, from_bus, to_bus):
    from_values = net.line["from_bus"].astype(int).to_numpy()
    to_values = net.line["to_bus"].astype(int).to_numpy()
    return ((from_values == int(from_bus)) & (to_values == int(to_bus))) | ((from_values == int(to_bus)) & (to_values == int(from_bus)))


def ensure_ieee33_tie_lines(net):
    ensure_line_columns(net)
    max_i_ka = float(pd.Series(net.line["max_i_ka"]).dropna().median()) if "max_i_ka" in net.line.columns else 0.40
    c_nf_per_km = float(pd.Series(net.line["c_nf_per_km"]).dropna().median()) if "c_nf_per_km" in net.line.columns else 0.0
    if not np.isfinite(max_i_ka) or max_i_ka <= 0 or max_i_ka > 5:
        max_i_ka = 0.40
    for tie in STANDARD_TIE_BRANCH_DATA:
        mask = line_mask_for_endpoints(net, tie["from_bus"], tie["to_bus"])
        matching = list(net.line.index[mask])
        if matching:
            net.line.loc[matching, "is_tie"] = True
            net.line.loc[matching, "in_service"] = False
            continue
        line_idx = pp.create_line_from_parameters(
            net,
            from_bus=tie["from_bus"],
            to_bus=tie["to_bus"],
            length_km=1.0,
            r_ohm_per_km=tie["r_ohm"],
            x_ohm_per_km=tie["x_ohm"],
            c_nf_per_km=c_nf_per_km,
            max_i_ka=max_i_ka,
            name=f"tie_{tie['from_bus'] + 1}_{tie['to_bus'] + 1}",
            in_service=False,
        )
        net.line.at[line_idx, "is_tie"] = True
    net.line["in_service"] = net.line["in_service"].astype(bool)
    net.line["is_tie"] = net.line["is_tie"].astype(bool)
    return net


def load_base_ieee33_for_edges():
    net = pn.case33bw()
    ensure_ieee33_tie_lines(net)
    endpoints = net.line[["from_bus", "to_bus"]].astype(int).to_numpy()
    return net, endpoints

BASE_NET_FOR_BENCHMARK, LINE_ENDPOINTS = load_base_ieee33_for_edges()
print(f"Physical line endpoints: {LINE_ENDPOINTS.shape}")


def active_edge_index_and_attr(edge_features: np.ndarray):
    active = edge_features[:, 2] > 0.5
    endpoints = LINE_ENDPOINTS[active]
    attrs = edge_features[active]
    if endpoints.shape[0] == 0:
        raise RuntimeError("Candidate has no active edges.")
    src = torch.tensor(endpoints[:, 0], dtype=torch.long)
    dst = torch.tensor(endpoints[:, 1], dtype=torch.long)
    edge_index = torch.stack([torch.cat([src, dst]), torch.cat([dst, src])], dim=0)
    edge_attr = torch.tensor(np.concatenate([attrs, attrs], axis=0), dtype=torch.float32)
    return edge_index, edge_attr


class H5CandidateDataset(Dataset):
    def __init__(self, h5_path: str, scenario_names: List[str], target_mean=None, target_std=None):
        self.h5_path = h5_path
        self.scenario_names = list(scenario_names)
        self.target_mean = None if target_mean is None else torch.tensor(target_mean, dtype=torch.float32)
        self.target_std = None if target_std is None else torch.tensor(target_std, dtype=torch.float32)
        self._h5 = None
        self.index = []
        with h5py.File(h5_path, "r") as h5:
            for scenario_pos, name in enumerate(self.scenario_names):
                n_candidates = int(h5[name]["candidate_labels"].shape[0])
                for candidate_pos in range(n_candidates):
                    self.index.append((scenario_pos, candidate_pos))

    def __len__(self):
        return len(self.index)

    def _file(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __getitem__(self, idx):
        scenario_pos, candidate_pos = self.index[idx]
        scenario_name = self.scenario_names[scenario_pos]
        group = self._file()[scenario_name]
        x = torch.tensor(group["candidate_node_features"][candidate_pos], dtype=torch.float32)
        edge_features = group["candidate_edge_features"][candidate_pos]
        edge_index, edge_attr = active_edge_index_and_attr(edge_features)
        u = torch.tensor(group["candidate_global_features"][candidate_pos], dtype=torch.float32).view(1, -1)
        full_label = torch.tensor(group["candidate_labels"][candidate_pos], dtype=torch.float32).view(1, -1)
        target_raw = full_label[:, :3]
        if self.target_mean is not None and self.target_std is not None:
            y = (target_raw - self.target_mean.view(1, -1)) / self.target_std.view(1, -1)
        else:
            y = target_raw
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, y=y)
        data.y_raw = target_raw
        data.y_full = full_label
        data.scenario_pos = torch.tensor([scenario_pos], dtype=torch.long)
        data.candidate_pos = torch.tensor([candidate_pos], dtype=torch.long)
        return data


def collect_targets(h5_path: str, scenario_names: List[str]):
    targets = []
    with h5py.File(h5_path, "r") as h5:
        for name in tqdm(scenario_names, desc="Collecting target stats", unit="scenario"):
            targets.append(h5[name]["candidate_labels"][:, :3])
    return np.concatenate(targets, axis=0).astype(np.float32)

TRAIN_TARGETS = collect_targets(DATA_PATH, TRAIN_SCENARIOS)
TARGET_MEAN = TRAIN_TARGETS.mean(axis=0)
TARGET_STD = TRAIN_TARGETS.std(axis=0) + 1e-8
print("Target mean:", dict(zip(TARGET_COLUMNS, TARGET_MEAN)))
print("Target std :", dict(zip(TARGET_COLUMNS, TARGET_STD)))

TRAIN_DATASET = H5CandidateDataset(DATA_PATH, TRAIN_SCENARIOS, TARGET_MEAN, TARGET_STD)
VAL_DATASET = H5CandidateDataset(DATA_PATH, VAL_SCENARIOS, TARGET_MEAN, TARGET_STD)
TEST_DATASET = H5CandidateDataset(DATA_PATH, TEST_SCENARIOS, TARGET_MEAN, TARGET_STD)

TRAIN_LOADER = DataLoader(TRAIN_DATASET, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
VAL_LOADER = DataLoader(VAL_DATASET, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
TEST_LOADER = DataLoader(TEST_DATASET, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

sample_data = TRAIN_DATASET[0]
NODE_DIM = sample_data.x.shape[-1]
EDGE_DIM = sample_data.edge_attr.shape[-1]
GLOBAL_DIM = sample_data.u.shape[-1]
print(f"Dataset sizes | train={len(TRAIN_DATASET):,}, val={len(VAL_DATASET):,}, test={len(TEST_DATASET):,}")
print(f"Feature dims | node={NODE_DIM}, edge={EDGE_DIM}, global={GLOBAL_DIM}")


## 3. Model definitions

All models output normalized predictions for `[loss, voltage, loading]`. Predictions are de-normalized for metrics and ranking.


In [ ]:
class GlobalMLP(nn.Module):
    def __init__(self, global_dim: int, hidden_dim: int = 128, out_dim: int = 3, dropout: float = 0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, batch):
        return self.net(batch.u)


class GCNRegressor(nn.Module):
    def __init__(self, node_dim: int, global_dim: int, hidden_dim: int = 128, layers: int = 3, out_dim: int = 3, dropout: float = 0.10):
        super().__init__()
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.convs = nn.ModuleList([GCNConv(hidden_dim, hidden_dim) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(layers)])
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2 + global_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, batch):
        x = self.node_encoder(batch.x)
        for conv, norm in zip(self.convs, self.norms):
            h = conv(x, batch.edge_index)
            h = F.relu(norm(h))
            x = F.dropout(h + x, p=self.dropout, training=self.training)
        pooled = torch.cat([global_mean_pool(x, batch.batch), global_max_pool(x, batch.batch)], dim=-1)
        return self.head(torch.cat([pooled, batch.u], dim=-1))


class GATRegressor(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, global_dim: int, hidden_dim: int = 128, heads: int = 4, layers: int = 3, out_dim: int = 3, dropout: float = 0.10):
        super().__init__()
        assert hidden_dim % heads == 0
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=edge_dim, dropout=dropout)
            for _ in range(layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(layers)])
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2 + global_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, batch):
        x = self.node_encoder(batch.x)
        for conv, norm in zip(self.convs, self.norms):
            h = conv(x, batch.edge_index, batch.edge_attr)
            h = F.relu(norm(h))
            x = F.dropout(h + x, p=self.dropout, training=self.training)
        pooled = torch.cat([global_mean_pool(x, batch.batch), global_max_pool(x, batch.batch)], dim=-1)
        return self.head(torch.cat([pooled, batch.u], dim=-1))


class GraphTransformerRegressor(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, global_dim: int, hidden_dim: int = 160, heads: int = 4, layers: int = 4, out_dim: int = 3, dropout: float = 0.10):
        super().__init__()
        assert hidden_dim % heads == 0
        self.node_encoder = nn.Sequential(nn.Linear(node_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.edge_encoder = nn.Sequential(nn.Linear(edge_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=hidden_dim, dropout=dropout, beta=True)
            for _ in range(layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(layers)])
        self.ffns = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim * 2, hidden_dim))
            for _ in range(layers)
        ])
        self.dropout = dropout
        self.global_encoder = nn.Sequential(nn.Linear(global_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Linear(hidden_dim // 2, out_dim),
        )

    def forward(self, batch):
        x = self.node_encoder(batch.x)
        edge_attr = self.edge_encoder(batch.edge_attr)
        for conv, norm, ffn in zip(self.convs, self.norms, self.ffns):
            h = conv(x, batch.edge_index, edge_attr)
            x = norm(x + F.dropout(h, p=self.dropout, training=self.training))
            x = norm(x + F.dropout(ffn(x), p=self.dropout, training=self.training))
        pooled = torch.cat([global_mean_pool(x, batch.batch), global_max_pool(x, batch.batch)], dim=-1)
        global_emb = self.global_encoder(batch.u)
        return self.head(torch.cat([pooled, global_emb], dim=-1))


def build_model(model_name: str):
    if model_name == "MLP":
        return GlobalMLP(GLOBAL_DIM)
    if model_name == "GCN":
        return GCNRegressor(NODE_DIM, GLOBAL_DIM)
    if model_name == "GAT":
        return GATRegressor(NODE_DIM, EDGE_DIM, GLOBAL_DIM)
    if model_name == "GraphTransformer":
        return GraphTransformerRegressor(NODE_DIM, EDGE_DIM, GLOBAL_DIM)
    raise ValueError(f"Unknown model: {model_name}")


## 4. Training utilities

Weighted MSE is applied in normalized target space. Regression metrics are reported in physical units.


In [ ]:
TARGET_MEAN_T = torch.tensor(TARGET_MEAN, dtype=torch.float32, device=DEVICE)
TARGET_STD_T = torch.tensor(TARGET_STD, dtype=torch.float32, device=DEVICE)
TARGET_WEIGHTS_T = TARGET_WEIGHTS.to(DEVICE)


def weighted_mse(pred_norm, target_norm):
    return ((pred_norm - target_norm) ** 2 * TARGET_WEIGHTS_T.view(1, -1)).mean()


def denormalize(pred_norm):
    return pred_norm * TARGET_STD_T.view(1, -1) + TARGET_MEAN_T.view(1, -1)


def move_batch(batch):
    return batch.to(DEVICE)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_count = 0
    for batch in loader:
        batch = move_batch(batch)
        if training:
            optimizer.zero_grad(set_to_none=True)
        pred = model(batch)
        loss = weighted_mse(pred, batch.y)
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        batch_size = int(batch.y.shape[0])
        total_loss += float(loss.detach().cpu()) * batch_size
        total_count += batch_size
    return total_loss / max(total_count, 1)


def train_model(model_name: str):
    model = build_model(model_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    history = []
    best_state = None
    best_val = float("inf")
    patience_left = PATIENCE
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(model, TRAIN_LOADER, optimizer)
        val_loss = run_epoch(model, VAL_LOADER, optimizer=None)
        scheduler.step(val_loss)
        history.append({"model": model_name, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"{model_name:16s} epoch={epoch:03d} train={train_loss:.6f} val={val_loss:.6f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping {model_name} at epoch {epoch}")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val


def predict_loader(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting", leave=False):
            batch = move_batch(batch)
            pred_norm = model(batch)
            pred = denormalize(pred_norm).detach().cpu().numpy()
            true = batch.y_raw.detach().cpu().numpy()
            full = batch.y_full.detach().cpu().numpy()
            scenario_pos = batch.scenario_pos.detach().cpu().numpy().reshape(-1)
            candidate_pos = batch.candidate_pos.detach().cpu().numpy().reshape(-1)
            for i in range(pred.shape[0]):
                rows.append({
                    "scenario_pos": int(scenario_pos[i]),
                    "candidate_pos": int(candidate_pos[i]),
                    "pred_loss": float(pred[i, 0]),
                    "pred_voltage": float(pred[i, 1]),
                    "pred_loading": float(pred[i, 2]),
                    "true_loss": float(true[i, 0]),
                    "true_voltage": float(true[i, 1]),
                    "true_loading": float(true[i, 2]),
                    "true_voltage_violations": int(full[i, 3]),
                    "true_overloads": int(full[i, 4]),
                    "true_constraint_violations": int(full[i, 5]),
                })
    return pd.DataFrame(rows)

MODEL_OBJECTS = {}
HISTORIES = []
BEST_VAL_LOSSES = {}
for model_name in MODELS_TO_TRAIN:
    print("=" * 80)
    print(f"Training {model_name}")
    model, history, best_val = train_model(model_name)
    MODEL_OBJECTS[model_name] = model
    HISTORIES.append(history)
    BEST_VAL_LOSSES[model_name] = best_val

TRAINING_HISTORY = pd.concat(HISTORIES, ignore_index=True) if HISTORIES else pd.DataFrame()


## 5. Regression and DNR ranking metrics


In [ ]:
def regression_metrics(df: pd.DataFrame, prefix=""):
    rows = []
    for target, pred_col, true_col in [
        ("loss", "pred_loss", "true_loss"),
        ("voltage", "pred_voltage", "true_voltage"),
        ("loading", "pred_loading", "true_loading"),
    ]:
        y = df[true_col].to_numpy(dtype=float)
        p = df[pred_col].to_numpy(dtype=float)
        err = p - y
        mae = float(np.mean(np.abs(err)))
        rmse = float(np.sqrt(np.mean(err ** 2)))
        denom = np.maximum(np.abs(y), 1e-8)
        mape = float(np.mean(np.abs(err) / denom) * 100.0)
        ss_res = float(np.sum((y - p) ** 2))
        ss_tot = float(np.sum((y - np.mean(y)) ** 2))
        r2 = float(1.0 - ss_res / max(ss_tot, 1e-12))
        rows.append({"target": target, "MAE": mae, "RMSE": rmse, "MAPE_percent": mape, "R2": r2})
    out = pd.DataFrame(rows)
    if prefix:
        out.insert(0, "model", prefix)
    return out


def rank_array(values, ascending=True):
    series = pd.Series(values)
    return series.rank(method="average", ascending=ascending).to_numpy(dtype=float)


def spearman_no_scipy(a, b):
    if len(a) < 2:
        return np.nan
    ra = rank_array(a, ascending=True)
    rb = rank_array(b, ascending=True)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return np.nan
    return float(np.corrcoef(ra, rb)[0, 1])


def ndcg_for_losses(true_losses, predicted_order):
    true_losses = np.asarray(true_losses, dtype=float)
    relevance = true_losses.max() - true_losses
    if np.allclose(relevance, 0):
        relevance = 1.0 / (1.0 + true_losses)
    gains = relevance[predicted_order]
    discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
    dcg = float(np.sum(gains * discounts))
    ideal = np.sort(relevance)[::-1]
    idcg = float(np.sum(ideal * discounts))
    return dcg / max(idcg, 1e-12)


def dnr_ranking_metrics(df: pd.DataFrame, model_name: str):
    metrics = []
    for scenario_pos, group in df.groupby("scenario_pos"):
        group = group.sort_values("candidate_pos").reset_index(drop=True)
        true_feasible = (
            (group["true_voltage"] >= FEASIBLE_VOLTAGE_MIN)
            & (group["true_loading"] <= FEASIBLE_LOADING_MAX)
            & (group["true_constraint_violations"] == 0)
        )
        pred_feasible = (
            (group["pred_voltage"] >= FEASIBLE_VOLTAGE_MIN)
            & (group["pred_loading"] <= FEASIBLE_LOADING_MAX)
        )
        true_candidates = group[true_feasible] if true_feasible.any() else group
        true_best_idx = int(true_candidates["true_loss"].idxmin())
        pred_candidates = group[pred_feasible] if pred_feasible.any() else group
        pred_best_idx = int(pred_candidates["pred_loss"].idxmin())
        pred_order = list(group.sort_values("pred_loss").index)
        true_order = list(group.sort_values("true_loss").index)
        true_rank_of_pred = int(true_order.index(pred_best_idx) + 1)
        top3 = set(pred_order[:3])
        metrics.append({
            "scenario_pos": int(scenario_pos),
            "model": model_name,
            "top1_correct": int(pred_best_idx == true_best_idx),
            "top3_correct": int(true_best_idx in top3),
            "mean_rank_error": abs(true_rank_of_pred - 1),
            "spearman": spearman_no_scipy(group["true_loss"].to_numpy(), group["pred_loss"].to_numpy()),
            "ndcg": ndcg_for_losses(group["true_loss"].to_numpy(), np.array(pred_order, dtype=int)),
            "average_loss_gap": float(group.loc[pred_best_idx, "true_loss"] - group.loc[true_best_idx, "true_loss"]),
            "true_best_idx": true_best_idx,
            "pred_best_idx": pred_best_idx,
            "true_best_loss": float(group.loc[true_best_idx, "true_loss"]),
            "selected_true_loss": float(group.loc[pred_best_idx, "true_loss"]),
        })
    scenario_metrics = pd.DataFrame(metrics)
    summary = {
        "model": model_name,
        "Top1_Accuracy": float(scenario_metrics["top1_correct"].mean()),
        "Top3_Accuracy": float(scenario_metrics["top3_correct"].mean()),
        "Mean_Rank_Error": float(scenario_metrics["mean_rank_error"].mean()),
        "Spearman": float(scenario_metrics["spearman"].mean(skipna=True)),
        "NDCG": float(scenario_metrics["ndcg"].mean()),
        "Average_Loss_Gap": float(scenario_metrics["average_loss_gap"].mean()),
    }
    return summary, scenario_metrics

PREDICTIONS = {}
REGRESSION_TABLES = []
RANKING_SUMMARIES = []
SCENARIO_RANKING_TABLES = []
for model_name, model in MODEL_OBJECTS.items():
    print(f"Evaluating {model_name}")
    pred_df = predict_loader(model, TEST_LOADER)
    PREDICTIONS[model_name] = pred_df
    REGRESSION_TABLES.append(regression_metrics(pred_df, model_name))
    rank_summary, scenario_rank_df = dnr_ranking_metrics(pred_df, model_name)
    RANKING_SUMMARIES.append(rank_summary)
    SCENARIO_RANKING_TABLES.append(scenario_rank_df)

REGRESSION_METRICS = pd.concat(REGRESSION_TABLES, ignore_index=True)
RANKING_METRICS = pd.DataFrame(RANKING_SUMMARIES)
SCENARIO_RANKING = pd.concat(SCENARIO_RANKING_TABLES, ignore_index=True)
display(REGRESSION_METRICS)
display(RANKING_METRICS)


## 6. Pandapower vs surrogate speed benchmark

This benchmark reconstructs candidate topologies from HDF5 metadata and reruns `pandapower.runpp()` for sampled test scenarios. Then it compares runtime to model forward passes on the same scenario sets.


In [ ]:
def read_metadata(dataset):
    raw = dataset[()]
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def apply_line_status_from_key(net, switch_config_key: str):
    active = set(int(x) for x in switch_config_key.split(",") if x != "")
    for line_idx in net.line.index:
        net.line.at[line_idx, "in_service"] = int(line_idx) in active
    return net


def apply_load_scaling(net, scaling):
    scaling = np.asarray(scaling, dtype=float)
    for load_idx, row in net.load.iterrows():
        scale = float(scaling[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale
    return net


def benchmark_pandapower_scenarios(h5_path: str, scenario_names: List[str]):
    candidate_count = 0
    start = time.perf_counter()
    with h5py.File(h5_path, "r") as h5:
        for scenario_name in scenario_names:
            group = h5[scenario_name]
            scaling = group["load_scaling_vector"][:]
            metadata = read_metadata(group["metadata"])
            for key in metadata["candidate_switch_config_keys"]:
                net = pn.case33bw()
                ensure_ieee33_tie_lines(net)
                apply_line_status_from_key(net, key)
                apply_load_scaling(net, scaling)
                pp.runpp(net, algorithm="nr", init="flat", calculate_voltage_angles=False, enforce_q_lims=False, max_iteration=50, tolerance_mva=1e-8, numba=False)
                candidate_count += 1
    elapsed = time.perf_counter() - start
    return {"pandapower_time": elapsed, "candidate_count": candidate_count, "pandapower_time_per_candidate": elapsed / max(candidate_count, 1)}


def make_loader_for_scenarios(scenario_names, batch_size=BATCH_SIZE):
    ds = H5CandidateDataset(DATA_PATH, scenario_names, TARGET_MEAN, TARGET_STD)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)


def benchmark_model_forward(model, scenario_names):
    loader = make_loader_for_scenarios(scenario_names)
    model.eval()
    count = 0
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch)
            _ = model(batch)
            count += int(batch.y.shape[0])
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {"model_time": elapsed, "candidate_count": count, "model_time_per_candidate": elapsed / max(count, 1)}


def run_speed_benchmarks(sample_sizes=(100, 500, 1000)):
    rng = np.random.default_rng(SEED)
    results = []
    available = np.array(TEST_SCENARIOS, dtype=object)
    for sample_size in sample_sizes:
        actual_size = min(int(sample_size), len(available))
        selected = list(rng.choice(available, size=actual_size, replace=False))
        pp_result = benchmark_pandapower_scenarios(DATA_PATH, selected)
        for model_name, model in MODEL_OBJECTS.items():
            model_result = benchmark_model_forward(model, selected)
            speedup = pp_result["pandapower_time"] / max(model_result["model_time"], 1e-12)
            results.append({
                "sample_size": actual_size,
                "model": model_name,
                "pandapower_time": pp_result["pandapower_time"],
                "model_time": model_result["model_time"],
                "candidate_count": model_result["candidate_count"],
                "pandapower_time_per_candidate": pp_result["pandapower_time_per_candidate"],
                "model_time_per_candidate": model_result["model_time_per_candidate"],
                "speedup": speedup,
            })
    return pd.DataFrame(results)

SPEED_BENCHMARKS = run_speed_benchmarks(sample_sizes=(100, 500, 1000))
display(SPEED_BENCHMARKS)
print("Average speedup:", SPEED_BENCHMARKS.groupby("model")["speedup"].mean().to_dict())
print("Median speedup :", SPEED_BENCHMARKS.groupby("model")["speedup"].median().to_dict())
print("Maximum speedup:", SPEED_BENCHMARKS.groupby("model")["speedup"].max().to_dict())


## 7. Plotly visualizations and scenario explorer


In [ ]:
def plot_training_curves(history_df):
    fig = go.Figure()
    for model_name, group in history_df.groupby("model"):
        fig.add_trace(go.Scatter(x=group["epoch"], y=group["train_loss"], mode="lines", name=f"{model_name} train"))
        fig.add_trace(go.Scatter(x=group["epoch"], y=group["val_loss"], mode="lines", name=f"{model_name} val"))
    fig.update_layout(title="Training Curves", xaxis_title="Epoch", yaxis_title="Weighted MSE")
    fig.show()


def plot_true_vs_pred(model_name="GraphTransformer"):
    df = PREDICTIONS[model_name]
    for true_col, pred_col, title in [
        ("true_loss", "pred_loss", "True vs Predicted Loss"),
        ("true_voltage", "pred_voltage", "True vs Predicted Voltage"),
        ("true_loading", "pred_loading", "True vs Predicted Loading"),
    ]:
        fig = px.scatter(df, x=true_col, y=pred_col, opacity=0.5, title=f"{model_name}: {title}")
        min_v = min(df[true_col].min(), df[pred_col].min())
        max_v = max(df[true_col].max(), df[pred_col].max())
        fig.add_trace(go.Scatter(x=[min_v, max_v], y=[min_v, max_v], mode="lines", name="ideal"))
        fig.show()


def plot_error_histograms(model_name="GraphTransformer"):
    df = PREDICTIONS[model_name].copy()
    df["loss_error"] = df["pred_loss"] - df["true_loss"]
    df["voltage_error"] = df["pred_voltage"] - df["true_voltage"]
    df["loading_error"] = df["pred_loading"] - df["true_loading"]
    melted = df[["loss_error", "voltage_error", "loading_error"]].melt(var_name="target", value_name="error")
    fig = px.histogram(melted, x="error", color="target", facet_row="target", nbins=80, title=f"{model_name}: Error Histograms")
    fig.show()


def plot_calibration(model_name="GraphTransformer", bins=20):
    df = PREDICTIONS[model_name].copy()
    plots = []
    for true_col, pred_col, target in [("true_loss", "pred_loss", "loss"), ("true_voltage", "pred_voltage", "voltage"), ("true_loading", "pred_loading", "loading")]:
        tmp = df[[true_col, pred_col]].copy()
        tmp["bin"] = pd.qcut(tmp[pred_col], q=min(bins, tmp[pred_col].nunique()), duplicates="drop")
        cal = tmp.groupby("bin", observed=True).agg(pred_mean=(pred_col, "mean"), true_mean=(true_col, "mean")).reset_index()
        cal["target"] = target
        plots.append(cal)
    cal_df = pd.concat(plots, ignore_index=True)
    fig = px.line(cal_df, x="pred_mean", y="true_mean", color="target", markers=True, title=f"{model_name}: Calibration Plots")
    fig.show()


def plot_model_comparisons():
    fig = px.bar(RANKING_METRICS, x="model", y="Top1_Accuracy", title="Topology Selection Top-1 Accuracy")
    fig.show()
    rank_melt = RANKING_METRICS.melt(id_vars="model", value_vars=["Top1_Accuracy", "Top3_Accuracy", "Spearman", "NDCG", "Average_Loss_Gap"], var_name="metric", value_name="value")
    fig = px.bar(rank_melt, x="model", y="value", color="metric", barmode="group", title="Ranking Metrics Dashboard")
    fig.show()
    speed = SPEED_BENCHMARKS.groupby("model", as_index=False)["speedup"].mean()
    fig = px.bar(speed, x="model", y="speedup", title="Average Speedup vs Pandapower")
    fig.show()


def scenario_explorer(model_name="GraphTransformer", max_scenarios=20):
    df = PREDICTIONS[model_name].copy()
    scenario_ids = sorted(df["scenario_pos"].unique())[:max_scenarios]
    fig = go.Figure()
    buttons = []
    traces_per_scenario = 4
    for idx, scenario_id in enumerate(scenario_ids):
        group = df[df["scenario_pos"] == scenario_id].sort_values("candidate_pos").reset_index(drop=True)
        true_best_pos = int(group["true_loss"].idxmin())
        pred_best_pos = int(group["pred_loss"].idxmin())
        visible = idx == 0
        fig.add_trace(go.Bar(
            x=group["candidate_pos"], y=group["true_loss"], name="True loss",
            marker_color="rgba(55, 83, 109, 0.55)", visible=visible,
        ))
        fig.add_trace(go.Scatter(
            x=group["candidate_pos"], y=group["pred_loss"], mode="lines+markers", name="Predicted loss",
            marker=dict(size=8), line=dict(width=3), visible=visible,
        ))
        fig.add_trace(go.Scatter(
            x=[group.loc[true_best_pos, "candidate_pos"]], y=[group.loc[true_best_pos, "true_loss"]],
            mode="markers", name="True best", marker=dict(size=16, symbol="star", color="green"), visible=visible,
        ))
        fig.add_trace(go.Scatter(
            x=[group.loc[pred_best_pos, "candidate_pos"]], y=[group.loc[pred_best_pos, "pred_loss"]],
            mode="markers", name="Predicted best", marker=dict(size=16, symbol="x", color="red"), visible=visible,
        ))
        visibility = [False] * (traces_per_scenario * len(scenario_ids))
        for trace_offset in range(traces_per_scenario):
            visibility[traces_per_scenario * idx + trace_offset] = True
        buttons.append({
            "label": f"scenario {scenario_id}",
            "method": "update",
            "args": [
                {"visible": visibility},
                {"title": f"Scenario Explorer: {model_name}, scenario_pos={scenario_id}"},
            ],
        })
    fig.update_layout(
        title=f"Scenario Explorer: {model_name}",
        xaxis_title="Candidate index",
        yaxis_title="Loss kW",
        updatemenus=[{"buttons": buttons, "direction": "down", "x": 1.02, "y": 1.0}],
        barmode="overlay",
    )
    fig.show()


plot_training_curves(TRAINING_HISTORY)
plot_true_vs_pred("GraphTransformer")
plot_error_histograms("GraphTransformer")
plot_calibration("GraphTransformer")
plot_model_comparisons()
scenario_explorer("GraphTransformer")



## 8. Final model comparison and conclusion


In [ ]:
def build_final_comparison_table():
    loss_reg = REGRESSION_METRICS[REGRESSION_METRICS["target"] == "loss"][["model", "MAE", "R2"]].rename(columns={"MAE": "MAE Loss", "R2": "R2 Loss"})
    rank = RANKING_METRICS[["model", "Top1_Accuracy", "NDCG", "Average_Loss_Gap"]]
    speed = SPEED_BENCHMARKS.groupby("model", as_index=False).agg(Inference_Time=("model_time_per_candidate", "mean"), Speedup_vs_Pandapower=("speedup", "mean"))
    table = loss_reg.merge(rank, on="model").merge(speed, on="model")
    table = table.sort_values(["NDCG", "Top1_Accuracy", "R2 Loss"], ascending=False).reset_index(drop=True)
    return table

FINAL_COMPARISON = build_final_comparison_table()
display(FINAL_COMPARISON)

best_model = FINAL_COMPARISON.iloc[0]["model"]
best_validation = BEST_VAL_LOSSES.get(best_model, np.nan)
best_test_reg = REGRESSION_METRICS[(REGRESSION_METRICS["model"] == best_model)]
best_rank = RANKING_METRICS[RANKING_METRICS["model"] == best_model].iloc[0].to_dict()
best_speedup = float(FINAL_COMPARISON.iloc[0]["Speedup_vs_Pandapower"])

print("DNR Surrogate Training Complete")
print("=" * 40)
print(f"Best Model              : {best_model}")
print(f"Best Validation Loss    : {best_validation:.6f}")
print("Test Regression Metrics :")
print(best_test_reg.to_string(index=False))
print("Ranking Metrics         :")
print(json.dumps({k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in best_rank.items()}, indent=2))
print(f"Pandapower Speedup      : {best_speedup:.2f}x")
if best_speedup > 10 and best_rank.get("NDCG", 0) > 0.90:
    suitability = "Suitable for fast candidate screening with periodic pandapower verification."
else:
    suitability = "Promising surrogate; validate on more scenarios before deployment."
print(f"Deployment suitability  : {suitability}")
